# Монте-Карло: BLER, доверительный интервал и потоки

Монте-Карло оценивает вероятность ошибки декодера по множеству случайных передач. В этом ноутбуке считаем точные frame errors, строим 95% интервалы Уилсона и измеряем пропускную способность пакетной обработки. Фиксированный seed делает пример повторяемым.

Интервал показывает неопределённость именно из-за конечного числа симуляций. Для инженерной оценки увеличивайте число кадров до тех пор, пока интервал не станет достаточно узким.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "Cargo.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Cargo.toml").exists():
    raise RuntimeError("Open this notebook from inside the ra-rust project")
sys.path.insert(0, str(ROOT / "python"))
from ra_reconcile import decode, decode_batch, encode
MATRIX_SMALL = ROOT / "data" / "H_small.edges"
MATRIX_RA = ROOT / "data" / "RA_n6000_R150.edges"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter

SEED = 731204
P = 0.01
FRAMES = 64
THREAD_COUNTS = [1, 2, 4]
MAX_ITER = 100
rng = np.random.default_rng(SEED)
k = 120
information = "".join(rng.integers(0, 2, size=k).astype(str))
codeword = encode(MATRIX_RA, information)
c = np.fromiter((int(bit) for bit in codeword), dtype=np.uint8)
bobs = rng.integers(0, 2, size=(FRAMES, len(c)), dtype=np.uint8)
errors = (rng.random((FRAMES, len(c))) < P).astype(np.uint8)
alice = bobs ^ errors
public = bobs ^ c
a_frames = ["".join(row.astype(str)) for row in alice]
m_frames = ["".join(row.astype(str)) for row in public]
print(f"Seed={SEED}; p={P}; frames={FRAMES}; n={len(c)}")

## Ошибка кадра и интервал Уилсона

Кадр считается ошибочным, если оценённое кодовое слово не совпало с исходным, даже если синдром равен нулю. Для `x` ошибок из `N` испытаний доля равна `x/N`. Интервал Уилсона устойчивее простого нормального интервала, особенно когда ошибок мало.


In [ ]:
def wilson_interval(errors, trials, z=1.96):
    phat = errors / trials
    den = 1 + z*z/trials
    center = (phat + z*z/(2*trials)) / den
    half = z*np.sqrt(phat*(1-phat)/trials + z*z/(4*trials*trials)) / den
    return max(0.0, center-half), min(1.0, center+half)

rows = []
for workers in THREAD_COUNTS:
    started = perf_counter()
    results = decode_batch(MATRIX_RA, a_frames, m_frames, P,
                           max_iter=MAX_ITER, threads=workers)
    wall_ms = (perf_counter() - started) * 1000
    failures = sum(r.estimate != codeword for r in results)
    lo, hi = wilson_interval(failures, FRAMES)
    rows.append({"threads": workers, "frames": FRAMES, "exact_frame_errors": failures,
                 "bler": failures/FRAMES, "bler_ci_low": lo, "bler_ci_high": hi,
                 "syndrome_failures": sum(not r.success for r in results),
                 "wall_ms": wall_ms, "frames_per_second": FRAMES/(wall_ms/1000)})
mc = pd.DataFrame(rows)
mc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mid = mc.bler.to_numpy()
axes[0].errorbar(mc.threads, mid,
                 yerr=[mid-mc.bler_ci_low, mc.bler_ci_high-mid],
                 fmt="o", capsize=5)
axes[0].set(xlabel="Потоки между кадрами", ylabel="BLER (95% CI)", ylim=(-.03, 1.03))
axes[0].grid(alpha=.3)
axes[1].bar(mc.threads.astype(str), mc.frames_per_second, color="steelblue")
axes[1].set(xlabel="Потоки между кадрами", ylabel="Кадров в секунду")
axes[1].grid(axis="y", alpha=.3)
fig.suptitle(f"Монте-Карло RA-6000: p={P}, N={FRAMES}")
fig.tight_layout()
plt.show()

## Что показывает эксперимент

При каждом числе потоков используется один и тот же фиксированный набор кадров, поэтому частота ошибок должна совпадать; меняется только длительность. На небольшом числе кадров один редкий сбой сильно меняет оценку BLER, и доверительный интервал будет широким. Увеличение `FRAMES` повышает статистическую точность. Скорость зависит от процессора, матрицы и числа итераций; потоков больше — не всегда быстрее из-за накладных расходов и ограничения пропускной способности памяти.
